# GraphRAG Unit-Test Pipeline: paragraph → graph → multi-hop answer → PDF

## 0. Import and configure

In [10]:
import os, re, json, hashlib
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_neo4j import Neo4jGraph

print("Imports loaded.")
print("Document = text + metadata; splitter = chunking; ChatOpenAI = model client;")
print("LLMGraphTransformer = graph extraction; Neo4jGraph = graph database connection.")


Imports loaded.
Document = text + metadata; splitter = chunking; ChatOpenAI = model client;
LLMGraphTransformer = graph extraction; Neo4jGraph = graph database connection.


In [2]:
api_key = os.getenv("LLM_API_KEY")
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

missing = [
    name for name, value in {
        "LLM_API_KEY": api_key,
        "NEO4J_URI": neo4j_uri,
        "NEO4J_USERNAME": neo4j_username,
        "NEO4J_PASSWORD": neo4j_password,
    }.items() if not value
]
if missing:
    raise ValueError("Set these environment variables before continuing: " + ", ".join(missing))

model_name = os.getenv("LLM_MODEL", "gpt-4o-mini")
llm_kwargs = {"model": model_name, "api_key": api_key, "temperature": 0}
if os.getenv("LLM_BASE_URL"):
    llm_kwargs["base_url"] = os.getenv("LLM_BASE_URL")

llm = ChatOpenAI(**llm_kwargs)
graph = Neo4jGraph(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_password,
)

print(f"LLM client configured: {model_name}")
print(f"Neo4j connection configured for: {neo4j_uri}")
print("Credentials are not displayed.")


LLM client configured: openai/gpt-oss-120b
Neo4j connection configured for: neo4j+s://cb96a564.databases.neo4j.io
Credentials are not displayed.


## Indexing Phase

## 1. Dataset IDs and helper functions

In [3]:
PARAGRAPH_DATASET_ID = "garden_paragraph_unit_test_v1"
PDF_DATASET_ID = "garden_pdf_unit_test_v1"

def normalize_name(value):
    value = str(value or "").casefold().strip()
    value = re.sub(r"[^\\w]+", " ", value, flags=re.UNICODE)
    return " ".join(value.split())

def safe_token(value, fallback="Entity"):
    token = re.sub(r"[^A-Za-z0-9_]", "_", str(value or ""))
    token = re.sub(r"_+", "_", token).strip("_")
    if not token or not token[0].isalpha():
        token = fallback
    return token[:60]

def entity_id(dataset_id, name):
    digest = hashlib.sha1(normalize_name(name).encode("utf-8")).hexdigest()[:12]
    return f"{dataset_id}::{digest}"

## 2. Data source: Paragraph with multiple entities and connected facts

In [4]:
story = (
    "Momna lives in Willow Village, a small community near Lahore. "
    "Momna grows tomatoes and spinach in the Willow Community Garden. "
    "Anfaal teaches Momna how to collect rainwater from the roof and store it in a clean tank. "
    "Momna uses the collected rainwater to water the community garden during dry weeks. "
    "The garden supplies fresh tomatoes and spinach to Willow Village School. "
    "Zainab is the cook at Willow Village School. "
    "Zainab uses the garden vegetables to prepare lunch for the school children. "
    "The school thanks the garden volunteers at its monthly community meeting."
)
source_doc = Document(
    page_content=story,
    metadata={"source": "garden_story", "document_id": "garden_story_001",
              "dataset_id": PARAGRAPH_DATASET_ID, "page": 1}
)
print(source_doc.page_content)
print("Metadata:", source_doc.metadata)

Momna lives in Willow Village, a small community near Lahore. Momna grows tomatoes and spinach in the Willow Community Garden. Anfaal teaches Momna how to collect rainwater from the roof and store it in a clean tank. Momna uses the collected rainwater to water the community garden during dry weeks. The garden supplies fresh tomatoes and spinach to Willow Village School. Zainab is the cook at Willow Village School. Zainab uses the garden vegetables to prepare lunch for the school children. The school thanks the garden volunteers at its monthly community meeting.
Metadata: {'source': 'garden_story', 'document_id': 'garden_story_001', 'dataset_id': 'garden_paragraph_unit_test_v1', 'page': 1}


## 3. Chunking with overlap and metadata

In [5]:
def chunk_documents(documents, chunk_size=260, chunk_overlap=90):
    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be smaller than chunk_size")
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        add_start_index=True,
        separators=["\\n\\n", "\\n", ". ", "; ", ", ", " ", ""],
        length_function=len,
    )
    chunks = splitter.split_documents(documents)
    for i, chunk in enumerate(chunks):
        chunk.metadata = dict(chunk.metadata)
        chunk.metadata["chunk_index"] = i
        chunk.metadata.setdefault("dataset_id", "unknown_dataset")
        chunk.metadata.setdefault("document_id", "unknown_document")
        chunk.metadata.setdefault("source", "unknown_source")
        chunk.metadata.setdefault("page", None)
    return chunks

paragraph_chunks = chunk_documents([source_doc], chunk_size=260, chunk_overlap=90)
print("Chunk count:", len(paragraph_chunks))
for chunk in paragraph_chunks:
    print("\\n---", chunk.metadata, "---")
    print(chunk.page_content)

Chunk count: 4
\n--- {'source': 'garden_story', 'document_id': 'garden_story_001', 'dataset_id': 'garden_paragraph_unit_test_v1', 'page': 1, 'start_index': 0, 'chunk_index': 0} ---
Momna lives in Willow Village, a small community near Lahore. Momna grows tomatoes and spinach in the Willow Community Garden. Anfaal teaches Momna how to collect rainwater from the roof and store it in a clean tank
\n--- {'source': 'garden_story', 'document_id': 'garden_story_001', 'dataset_id': 'garden_paragraph_unit_test_v1', 'page': 1, 'start_index': 125, 'chunk_index': 1} ---
. Anfaal teaches Momna how to collect rainwater from the roof and store it in a clean tank. Momna uses the collected rainwater to water the community garden during dry weeks. The garden supplies fresh tomatoes and spinach to Willow Village School
\n--- {'source': 'garden_story', 'document_id': 'garden_story_001', 'dataset_id': 'garden_paragraph_unit_test_v1', 'page': 1, 'start_index': 298, 'chunk_index': 2} ---
. The garden supplie

## 4. Convert each chunk into graph documents

In [ ]:
transformer = LLMGraphTransformer(llm=llm)
paragraph_graph_docs = transformer.convert_to_graph_documents(paragraph_chunks)

for i, gd in enumerate(paragraph_graph_docs):
    print(f"\\nCHUNK {i}:")
    print("NODES:")
    for node in gd.nodes:
        print({"id": node.id, "type": node.type, "properties": node.properties})
    print("RELATIONSHIPS:")
    for rel in gd.relationships:
        print({"source": rel.source.id, "type": rel.type, "target": rel.target.id,
               "properties": rel.properties})

\nCHUNK 0:
NODES:
{'id': 'Momna', 'type': 'Person', 'properties': {}}
{'id': 'Willow Village', 'type': 'Place', 'properties': {}}
{'id': 'Lahore', 'type': 'Place', 'properties': {}}
{'id': 'Willow Community Garden', 'type': 'Place', 'properties': {}}
{'id': 'Anfaal', 'type': 'Person', 'properties': {}}
RELATIONSHIPS:
{'source': 'Momna', 'type': 'LIVES_IN', 'target': 'Willow Village', 'properties': {}}
{'source': 'Willow Village', 'type': 'NEAR', 'target': 'Lahore', 'properties': {}}
{'source': 'Momna', 'type': 'GROWS_IN', 'target': 'Willow Community Garden', 'properties': {}}
{'source': 'Anfaal', 'type': 'TEACHES', 'target': 'Momna', 'properties': {}}
\nCHUNK 1:
NODES:
{'id': 'Anfaal', 'type': 'Person', 'properties': {}}
{'id': 'Momna', 'type': 'Person', 'properties': {}}
{'id': 'Rainwater', 'type': 'Concept', 'properties': {}}
{'id': 'Roof', 'type': 'Concept', 'properties': {}}
{'id': 'Clean Tank', 'type': 'Concept', 'properties': {}}
{'id': 'Community Garden', 'type': 'Location', 'pr

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to multipart ingest runs: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:2417)')))"))
Content-Length: 301066
API Key: lsv2_********************************************4atrace=01a19896-4b13-76b0-beae-7299fd77b22d,id=01a19896-4b13-76b0-beae-7299fd77b22d; trace=01a19896-4b13-76b0-beae-7299fd77b22d,id=01a19896-4b2c-7220-a71e-c7030e13929c; trace=01a19896-4b13-76b0-beae-7299fd77b22d,id=01a19896-4b2d-7

## 5. Entity resolution 


In [11]:
def prepare_rows(graph_docs, chunks, dataset_id):
    nodes_by_key = {}
    edges = []
    for i, gd in enumerate(graph_docs):
        chunk = chunks[i] if i < len(chunks) else None
        meta = dict(chunk.metadata) if chunk else {}
        chunk_text = chunk.page_content if chunk else ""
        for node in gd.nodes:
            name = str(node.id).strip()
            norm = normalize_name(name)
            if not norm:
                continue
            key = (dataset_id, norm)
            props = dict(getattr(node, "properties", {}) or {})
            desc = str(props.get("description") or props.get("summary") or "")
            entry = nodes_by_key.setdefault(key, {
                "dataset_id": dataset_id, "entity_id": entity_id(dataset_id, name),
                "name": name, "normalized_name": norm,
                "entity_type": safe_token(getattr(node, "type", "Entity")),
                "description": desc, "source_ids": set(), "aliases": set()
            })
            entry["source_ids"].add(str(meta.get("chunk_index", i)))
            if not entry["description"] and desc:
                entry["description"] = desc
            if name != entry["name"]:
                entry["aliases"].add(name)
        for rel in gd.relationships:
            source_name, target_name = str(rel.source.id).strip(), str(rel.target.id).strip()
            if not source_name or not target_name:
                continue
            edges.append({
                "dataset_id": dataset_id,
                "source_id": entity_id(dataset_id, source_name), "source_name": source_name,
                "source_type": safe_token(getattr(rel.source, "type", "Entity")),
                "target_id": entity_id(dataset_id, target_name), "target_name": target_name,
                "target_type": safe_token(getattr(rel.target, "type", "Entity")),
                "relation_type": safe_token(getattr(rel, "type", "RELATED_TO"), "RELATED_TO").upper(),
                "document_id": str(meta.get("document_id", "unknown_document")),
                "source": str(meta.get("source", "unknown_source")),
                "page": meta.get("page"), "chunk_id": str(meta.get("chunk_index", i)),
                "source_text": chunk_text,
                "properties_json": json.dumps(dict(getattr(rel, "properties", {}) or {},), ensure_ascii=False)
            })
    nodes = []
    for n in nodes_by_key.values():
        n["source_ids"] = sorted(n["source_ids"])
        n["aliases"] = sorted(n["aliases"])
        nodes.append(n)
    return nodes, edges

paragraph_nodes, paragraph_edges = prepare_rows(
    paragraph_graph_docs, paragraph_chunks, PARAGRAPH_DATASET_ID
)
print("Unique normalized entities:", len(paragraph_nodes))
print("Relationship records:", len(paragraph_edges))
print("\\nENTITIES:")
for n in paragraph_nodes:
    print({k: v for k, v in n.items()})
print("\\nEDGES:")
for e in paragraph_edges:
    print(f"{e['source_name']} --{e['relation_type']}--> {e['target_name']} (chunk {e['chunk_id']})")

Unique normalized entities: 2
Relationship records: 25
\nENTITIES:
{'dataset_id': 'garden_paragraph_unit_test_v1', 'entity_id': 'garden_paragraph_unit_test_v1::202b303291a8', 'name': 'Willow Village', 'normalized_name': 'w w', 'entity_type': 'Place', 'description': '', 'source_ids': ['0', '1', '2'], 'aliases': ['Willow Community Garden', 'Willow Village School']}
{'dataset_id': 'garden_paragraph_unit_test_v1', 'entity_id': 'garden_paragraph_unit_test_v1::aff024fe4ab0', 'name': 'Rainwater', 'normalized_name': 'w', 'entity_type': 'Concept', 'description': '', 'source_ids': ['1'], 'aliases': []}
\nEDGES:
Momna --LIVES_IN--> Willow Village (chunk 0)
Willow Village --NEAR--> Lahore (chunk 0)
Momna --GROWS_IN--> Willow Community Garden (chunk 0)
Anfaal --TEACHES--> Momna (chunk 0)
Anfaal --TEACHES--> Momna (chunk 1)
Momna --COLLECTS--> Rainwater (chunk 1)
Rainwater --COLLECTED_FROM--> Roof (chunk 1)
Rainwater --STORED_IN--> Clean Tank (chunk 1)
Momna --USES--> Rainwater (chunk 1)
Momna --WAT

## 6. Store nodes and edges in Neo4j

In [18]:
# 1. Inspect what the LLM actually extracted.
for gd in paragraph_graph_docs:
    print("NODES:")
    for node in gd.nodes:
        print(node.id, node.type, node.properties)

    print("RELATIONSHIPS:")
    for rel in gd.relationships:
        print(rel.source.id, rel.type, rel.target.id, rel.properties)

# 2. Add dataset metadata before writing.
for gd in paragraph_graph_docs:
    for node in gd.nodes:
        node.properties["dataset_id"] = PARAGRAPH_DATASET_ID

    for rel in gd.relationships:
        rel.properties["dataset_id"] = PARAGRAPH_DATASET_ID

# 3. Save using LangChain's built-in method.
graph.add_graph_documents(paragraph_graph_docs)

NODES:
Momna Person {}
Willow Village Place {}
Lahore Place {}
Willow Community Garden Place {}
Anfaal Person {}
RELATIONSHIPS:
Momna LIVES_IN Willow Village {}
Willow Village NEAR Lahore {}
Momna GROWS_IN Willow Community Garden {}
Anfaal TEACHES Momna {}
NODES:
Anfaal Person {}
Momna Person {}
Rainwater Concept {}
Roof Concept {}
Clean Tank Concept {}
Community Garden Location {}
Fresh Tomatoes Concept {}
Spinach Concept {}
Willow Village School Organization {}
RELATIONSHIPS:
Anfaal TEACHES Momna {}
Momna COLLECTS Rainwater {}
Rainwater COLLECTED_FROM Roof {}
Rainwater STORED_IN Clean Tank {}
Momna USES Rainwater {}
Momna WATER Community Garden {}
Community Garden SUPPLIES Fresh Tomatoes {}
Community Garden SUPPLIES Spinach {}
Fresh Tomatoes SUPPLIED_TO Willow Village School {}
Spinach SUPPLIED_TO Willow Village School {}
NODES:
Garden Place {}
Tomatoes Food {}
Spinach Food {}
Willow Village School Organization {}
Zainab Person {}
RELATIONSHIPS:
Garden SUPPLIES Tomatoes {}
Garden SUP

In [22]:
# Inspect nodes belonging to this dataset
node_rows = graph.query(
    """
    MATCH (n)
    WHERE n.dataset_id = $dataset_id
    RETURN
        elementId(n) AS neo4j_id,
        n.id AS id,
        labels(n) AS labels,
        properties(n) AS properties
    ORDER BY labels(n), n.id
    """,
    {"dataset_id": PARAGRAPH_DATASET_ID}
)

# Inspect relationships belonging to this dataset
edge_rows = graph.query(
    """
    MATCH (a)-[r]->(b)
    WHERE a.dataset_id = $dataset_id
      AND b.dataset_id = $dataset_id
      AND r.dataset_id = $dataset_id
    RETURN
        a.id AS source,
        type(r) AS relationship,
        b.id AS target,
        properties(r) AS properties
    ORDER BY source, relationship, target
    """,
    {"dataset_id": PARAGRAPH_DATASET_ID}
)

print("NODES:", len(node_rows))
print(json.dumps(node_rows, indent=2, ensure_ascii=False))

print("\nEDGES:", len(edge_rows))
print(json.dumps(edge_rows, indent=2, ensure_ascii=False))

NODES: 21
[
  {
    "neo4j_id": "4:01558d86-7ce2-4b8a-af70-5b2428bcc5fe:20",
    "id": "Clean Tank",
    "labels": [
      "Concept"
    ],
    "properties": {
      "dataset_id": "garden_paragraph_unit_test_v1",
      "id": "Clean Tank"
    }
  },
  {
    "neo4j_id": "4:01558d86-7ce2-4b8a-af70-5b2428bcc5fe:1844",
    "id": "Fresh Tomatoes",
    "labels": [
      "Concept"
    ],
    "properties": {
      "dataset_id": "garden_paragraph_unit_test_v1",
      "id": "Fresh Tomatoes"
    }
  },
  {
    "neo4j_id": "4:01558d86-7ce2-4b8a-af70-5b2428bcc5fe:18",
    "id": "Rainwater",
    "labels": [
      "Concept"
    ],
    "properties": {
      "dataset_id": "garden_paragraph_unit_test_v1",
      "id": "Rainwater"
    }
  },
  {
    "neo4j_id": "4:01558d86-7ce2-4b8a-af70-5b2428bcc5fe:19",
    "id": "Roof",
    "labels": [
      "Concept"
    ],
    "properties": {
      "dataset_id": "garden_paragraph_unit_test_v1",
      "id": "Roof"
    }
  },
  {
    "neo4j_id": "4:01558d86-7ce2-4b8a-af

## Reterieval Phase

## 7. Extract query entities

In [37]:
# question = (
#     "How does Anfaal's teaching help the school prepare lunch for the children?"
# )
question = (
    "How does zainab able to prepare lunch for the school children?"
)

In [38]:
from pydantic import BaseModel, Field

class QueryEntities(BaseModel):
    entities: list[str] = Field(
        description="Individual searchable people, places, organizations, "
                    "objects, and concepts mentioned or implied in the question."
    )

entity_extractor = llm.with_structured_output(QueryEntities)

query_result = entity_extractor.invoke([
    (
        "system",
        """
        Extract individual search entities from the question.

        Rules:
        1. Split possessive phrases into their underlying entities.
        2. Extract people, places, organizations, objects and concepts.
        3. Include important subjects such as lunch and school children.
        4. Return individual search terms, not long phrases.
        5. Do not answer the question or invent information.
        """
    ),
    ("human", question)
])

query_entities = list(dict.fromkeys(
    e.strip() for e in query_result.entities if e.strip()
))

print("Question:", question)
print("Extracted search entities:", query_entities)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Question: How does zainab able to prepare lunch for the school children?
Extracted search entities: ['zainab', 'lunch', 'school children']


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


## 8. Match Entities

In [39]:
#find the matched entities in the stored entity names
matched_entities = []

for entity in query_entities:
    results = graph.query(
        """
        MATCH (n)
        WHERE n.dataset_id = $dataset_id
          AND n.id IS NOT NULL
          AND toLower(n.id) CONTAINS toLower($entity)
        RETURN
            elementId(n) AS node_id,
            n.id AS name,
            labels(n) AS labels,
            properties(n) AS properties
        LIMIT 10
        """,
        {
            "dataset_id": PARAGRAPH_DATASET_ID,
            "entity": entity
        }
    )

    for result in results:
        matched_entities.append({
            "query_term": entity,
            "node_id": result["node_id"],
            "name": result["name"],
            "labels": result["labels"]
        })

print("Matched graph entities:")
print(json.dumps(matched_entities, indent=2, ensure_ascii=False))

Matched graph entities:
[
  {
    "query_term": "zainab",
    "node_id": "4:01558d86-7ce2-4b8a-af70-5b2428bcc5fe:1850",
    "name": "Zainab",
    "labels": [
      "Person"
    ]
  },
  {
    "query_term": "lunch",
    "node_id": "4:01558d86-7ce2-4b8a-af70-5b2428bcc5fe:1852",
    "name": "Lunch",
    "labels": [
      "Food"
    ]
  },
  {
    "query_term": "school children",
    "node_id": "4:01558d86-7ce2-4b8a-af70-5b2428bcc5fe:1853",
    "name": "School Children",
    "labels": [
      "Person"
    ]
  }
]


## 9. Local graph retrieval with configurable hops

In [43]:
# graph traversal
MAX_HOPS = 3 

matched_node_ids = list(dict.fromkeys(
    item["node_id"] for item in matched_entities
))

if not matched_node_ids:
    raise ValueError("No query entities matched nodes in Neo4j.")

traversal_results = graph.query(
    f"""
    MATCH (seed)
    WHERE elementId(seed) IN $node_ids
      AND seed.dataset_id = $dataset_id

    MATCH path = (seed)-[*1..{MAX_HOPS}]-(neighbor)

    WHERE all(
        n IN nodes(path)
        WHERE n.dataset_id = $dataset_id
    )

    RETURN
        [n IN nodes(path) |
            {{name: n.id, labels: labels(n)}}
        ] AS path_nodes,

        [r IN relationships(path) |
            {{type: type(r), properties: properties(r)}}
        ] AS path_relationships

    LIMIT 30
    """,
    {
        "node_ids": matched_node_ids,
        "dataset_id": PARAGRAPH_DATASET_ID
    }
)

print("Traversal paths:", len(traversal_results))
print(json.dumps(
    traversal_results,
    indent=2,
    ensure_ascii=False,
    default=str
))

Traversal paths: 30
[
  {
    "path_nodes": [
      {
        "labels": [
          "Person"
        ],
        "name": "Zainab"
      },
      {
        "labels": [
          "Place"
        ],
        "name": "Garden"
      }
    ],
    "path_relationships": [
      {
        "properties": {
          "dataset_id": "garden_paragraph_unit_test_v1"
        },
        "type": "USES"
      }
    ]
  },
  {
    "path_nodes": [
      {
        "labels": [
          "Person"
        ],
        "name": "Zainab"
      },
      {
        "labels": [
          "Place"
        ],
        "name": "Garden"
      },
      {
        "labels": [
          "Organization"
        ],
        "name": "Willow Village School"
      }
    ],
    "path_relationships": [
      {
        "properties": {
          "dataset_id": "garden_paragraph_unit_test_v1"
        },
        "type": "USES"
      },
      {
        "properties": {
          "dataset_id": "garden_paragraph_unit_test_v1"
        },
        "typ

## 10. Show Reterived Facts

In [44]:
#passing facts
graph_context = []

for path in traversal_results:
    nodes = path["path_nodes"]
    relationships = path["path_relationships"]

    for i, rel in enumerate(relationships):
        source = nodes[i]["name"]
        target = nodes[i + 1]["name"]

        graph_context.append(
            f"{source} --[{rel['type']}]--> {target}"
        )

# Remove duplicate facts
graph_context = list(dict.fromkeys(graph_context))

print("Retrieved facts:")
for fact in graph_context:
    print("-", fact)

Retrieved facts:
- Zainab --[USES]--> Garden
- Garden --[SUPPLIES]--> Willow Village School
- Willow Village School --[SUPPLIED_TO]--> Fresh Tomatoes
- Willow Village School --[SUPPLIED_TO]--> Spinach
- Willow Village School --[COOK]--> Zainab
- Garden --[SUPPLIES]--> Tomatoes
- Tomatoes --[USES]--> Zainab
- Garden --[SUPPLIES]--> Spinach
- Spinach --[USES]--> Zainab
- Zainab --[USES]--> Tomatoes
- Tomatoes --[SUPPLIES]--> Garden
- Garden --[USES]--> Zainab
- Zainab --[USES]--> Spinach
- Spinach --[SUPPLIES]--> Garden
- Zainab --[USES]--> Garden Vegetables
- Zainab --[PREPARES]--> Lunch
- Lunch --[FOR]--> School Children
- Zainab --[COOK]--> Willow Village School
- Fresh Tomatoes --[SUPPLIES]--> Community Garden
- Spinach --[SUPPLIES]--> Community Garden
- Willow Village School --[SUPPLIES]--> Garden


## 11. Generate Response (pass query and reterived facts)

In [45]:
#generat answer
if not graph_context:
    print("No connected graph facts found.")
else:
    context = "\n".join(graph_context)

    answer_prompt = f"""
    You are answering a question using retrieved knowledge graph facts.

    Retrieved facts:
    {context}

    Question:
    {question}

    Instructions:
    - Answer using only the retrieved facts.
    - Explain the relationships between the relevant entities.
    - Do not invent missing information.
    - If the facts are insufficient, clearly say so.
    """

    response = llm.invoke(answer_prompt)

    print("Answer:")
    print(response.content)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Answer:
Zainab is able to prepare lunch for the school children because the knowledge‑graph shows a chain of existing relationships that connect her, the food ingredients, the school, and the lunch itself:

1. **Zainab prepares lunch** – the fact *“Zainab --[PREPARES]--> Lunch”* states directly that she is the one who makes the lunch.  
2. **The lunch is for school children** – the fact *“Lunch --[FOR]--> School Children”* tells us the purpose of the lunch she prepares.  

The way she obtains the food needed for that lunch is also described by the facts:

3. **Zainab uses the garden and its produce** – she has the relationships  
   * “Zainab --[USES]--> Garden”  
   * “Zainab --[USES]--> Tomatoes”  
   * “Zainab --[USES]--> Spinach”  
   * “Zainab --[USES]--> Garden Vegetables”.  

4. **The garden supplies the vegetables** – the garden provides the raw ingredients:  
   * “Garden --[SUPPLIES]--> Tomatoes”  
   * “Garden --[SUPPLIES]--> Spinach”.  

5. **The garden (and its produce) su

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


## Multi-entity / multi-hop Questions

1. Direct fact: `Who teaches Momna?`
2. Multi-entity: `What connects Momna, the garden, and Willow Village School?`
3. Multi-hop: `How does Anfaal's teaching relate to lunch at the school?`
4. Unsupported: `What is the principal's name?`

## Testing with a pdf: Atomic habits

In [46]:
from pathlib import Path
from pypdf import PdfReader
from langchain_core.documents import Document

PDF_PATH = Path("/home/momina/Downloads/atomic-habbits-chp1.pdf")
PDF_DATASET_ID = "atomic-habits-pdf_unit_test_v1"

assert PDF_PATH.exists(), f"PDF not found: {PDF_PATH}"
assert PDF_PATH.suffix.lower() == ".pdf", "Expected a PDF file"

## pdf page extraction and Chunking

In [47]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)


reader = PdfReader(str(PDF_PATH))
pdf_documents = []

for page_number, page in enumerate(reader.pages, start=1):
    text = (page.extract_text() or "").strip()

    if text:
        pdf_documents.append(
            Document(
                page_content=text,
                metadata={
                    "source": PDF_PATH.name,
                    "dataset_id": PDF_DATASET_ID,
                    "page": page_number,
                },
            )
        )

pdf_chunks = text_splitter.split_documents(pdf_documents)

for index, chunk in enumerate(pdf_chunks):
    chunk.metadata["chunk_index"] = index

print("Pages with text:", len(pdf_documents))
print("Total chunks:", len(pdf_chunks))
print("First chunk metadata:", pdf_chunks[0].metadata if pdf_chunks else None)

Pages with text: 12
Total chunks: 42
First chunk metadata: {'source': 'atomic-habbits-chp1.pdf', 'dataset_id': 'atomic-habits-pdf_unit_test_v1', 'page': 1, 'chunk_index': 0}


## Extract Graph

In [48]:
pdf_graph_docs = transformer.convert_to_graph_documents(pdf_chunks)

print("Graph documents:", len(pdf_graph_docs))
print("Extracted nodes:", sum(len(gd.nodes) for gd in pdf_graph_docs))
print(
    "Extracted relationships:",
    sum(len(gd.relationships) for gd in pdf_graph_docs)
)

for index, gd in enumerate(pdf_graph_docs[:3]):
    print(f"\nChunk {index}")

    print("Nodes:")
    for node in gd.nodes:
        print(node.id, node.type, node.properties)

    print("Relationships:")
    for rel in gd.relationships:
        print(rel.source.id, "--", rel.type, "-->", rel.target.id)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTP

Graph documents: 42
Extracted nodes: 448
Extracted relationships: 278

Chunk 0
Nodes:
British Cycling Organization {}
Dave Brailsford Person {}
Great Britain Location {}
2003 Date {}
1908 Date {}
Olympic Games Event {}
Tour De France Event {}
Performance Director Position {}
Relationships:
Dave Brailsford -- EMPLOYED_BY --> British Cycling
British Cycling -- GOVERNING_BODY_FOR --> Great Britain
Dave Brailsford -- HELD_POSITION --> Performance Director
Performance Director -- POSITION_AT --> British Cycling

Chunk 1
Nodes:
Top Bike Manufacturer Organization {}
Team Organization {}
British Cycling Organization {}
Brailsford Person {}
Relationships:
Top Bike Manufacturer -- REFUSED --> Team
Brailsford -- HIRED_FOR --> British Cycling

Chunk 2
Nodes:
Brailsford Person {}
Coaches Person {}
Professional Cycling Team Organization {}
Bike Seats Object {}
Alcohol Substance {}
Tires Object {}
Riders Person {}
Electrically Heated Overshorts Object {}
Muscle Temperature Concept {}
Biofeedback Sens

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


## Add Graph Docs to Neo4j with metadata

In [49]:
def prepare_pdf_graph_documents(graph_documents, dataset_id):
    prepared = []

    for gd in graph_documents:
        # Copy the document metadata so the original isn't modified.
        source_metadata = dict(gd.source.metadata or {})
        source_metadata["dataset_id"] = dataset_id

        # Keep the source document and its provenance.
        source_document = Document(
            page_content=gd.source.page_content,
            metadata=source_metadata,
        )

        # Namespace entity IDs to avoid merging with other datasets.
        for node in gd.nodes:
            original_id = str(node.id)
            node.properties = dict(node.properties or {})
            node.properties["display_name"] = (
                node.properties.get("name") or original_id
            )
            node.properties["original_entity_id"] = original_id
            node.properties["dataset_id"] = dataset_id
            node.id = f"{dataset_id}::{original_id}"

        for rel in gd.relationships:
            rel.properties = dict(rel.properties or {})
            rel.properties["dataset_id"] = dataset_id

            # Relationship endpoints must use the same namespaced IDs
            # as the nodes in this graph document.

        prepared.append(
            type(gd)(
                nodes=gd.nodes,
                relationships=gd.relationships,
                source=source_document,
            )
        )

    return prepared


prepared_pdf_graph_docs = prepare_pdf_graph_documents(
    pdf_graph_docs,
    PDF_DATASET_ID,
)

assert prepared_pdf_graph_docs, "No PDF graph documents to store"

graph.add_graph_documents(
    prepared_pdf_graph_docs,
    include_source=True,
)

print("PDF graph written to Neo4j.")
print("Dataset:", PDF_DATASET_ID)

PDF graph written to Neo4j.
Dataset: atomic-habits-pdf_unit_test_v1


In [50]:
pdf_node_rows = graph.query(
    """
    MATCH (n)
    WHERE n.dataset_id = $dataset_id
    RETURN
        n.id AS id,
        n.display_name AS name,
        labels(n) AS labels,
        properties(n) AS properties
    ORDER BY name
    """,
    {"dataset_id": PDF_DATASET_ID},
)

pdf_edge_rows = graph.query(
    """
    MATCH (a)-[r]->(b)
    WHERE a.dataset_id = $dataset_id
      AND b.dataset_id = $dataset_id
      AND r.dataset_id = $dataset_id
    RETURN
        a.display_name AS source,
        type(r) AS relationship,
        b.display_name AS target,
        properties(r) AS properties
    ORDER BY source, relationship, target
    """,
    {"dataset_id": PDF_DATASET_ID},
)

print("PDF nodes:", len(pdf_node_rows))
print("PDF relationships:", len(pdf_edge_rows))

print("\nNodes:")
print(json.dumps(pdf_node_rows[:20], indent=2, ensure_ascii=False, default=str))

print("\nRelationships:")
print(json.dumps(pdf_edge_rows[:20], indent=2, ensure_ascii=False, default=str))

PDF nodes: 403
PDF relationships: 0

Nodes:
[
  {
    "id": "atomic-habits-pdf_unit_test_v1::1 Percent",
    "name": "1 Percent",
    "labels": [
      "Concept"
    ],
    "properties": {
      "dataset_id": "atomic-habits-pdf_unit_test_v1",
      "original_entity_id": "1 Percent",
      "id": "atomic-habits-pdf_unit_test_v1::1 Percent",
      "display_name": "1 Percent"
    }
  },
  {
    "id": "atomic-habits-pdf_unit_test_v1::1 Percent Error",
    "name": "1 Percent Error",
    "labels": [
      "Concept"
    ],
    "properties": {
      "dataset_id": "atomic-habits-pdf_unit_test_v1",
      "original_entity_id": "1 Percent Error",
      "id": "atomic-habits-pdf_unit_test_v1::1 Percent Error",
      "display_name": "1 Percent Error"
    }
  },
  {
    "id": "atomic-habits-pdf_unit_test_v1::1% Better Each Day",
    "name": "1% Better Each Day",
    "labels": [
      "Concept"
    ],
    "properties": {
      "dataset_id": "atomic-habits-pdf_unit_test_v1",
      "original_entity_id": "

## Cleanup query (Dataset Deletion)

In [ ]:

# graph.query("""
# MATCH (n:GraphEntity {dataset_id: $dataset_id})
# DETACH DELETE n
# """, {"dataset_id": PARAGRAPH_DATASET_ID})